In [1]:
# cell 1
# Mount Google Drive and define project paths.

from google.colab import drive
drive.mount("/content/drive")

import os
import sys
import json
import random
import subprocess
from pathlib import Path

DRIVE_ROOT = "/content/drive/MyDrive"
PROJECT_DIR = f"{DRIVE_ROOT}/final_project"

PYRAG_WORK_DIR = f"{PROJECT_DIR}/pyrag"
REPO_DIR = "/content/PyRAG"
INDEX_ROOT = f"{PYRAG_WORK_DIR}/indexes"
PREPARED_Q_DIR = f"{PYRAG_WORK_DIR}/prepared_questions"
CUSTOM_SCRIPTS_DIR = f"{PYRAG_WORK_DIR}/custom_scripts"

for p in [PYRAG_WORK_DIR, INDEX_ROOT, PREPARED_Q_DIR, CUSTOM_SCRIPTS_DIR]:
    Path(p).mkdir(parents=True, exist_ok=True)

print("Project dir:", PROJECT_DIR)
print("PyRAG work dir:", PYRAG_WORK_DIR)
print("Recommended runtime for the full pipeline: A100 80GB")

# Show GPU information.
subprocess.run(["nvidia-smi"], check=False)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project dir: /content/drive/MyDrive/final_project
PyRAG work dir: /content/drive/MyDrive/final_project/pyrag
Recommended runtime for the full pipeline: A100 80GB


CompletedProcess(args=['nvidia-smi'], returncode=0)

In [2]:
# cell 2
# Install only the dependencies needed for Phase 1:
# - sentence-transformers for E5 embeddings
# - faiss-cpu for local dense retrieval index
# - fastapi/uvicorn for a PyRAG-compatible retriever server

import sys
import subprocess

packages = [
    "sentence-transformers>=3.0.1",
    "faiss-cpu>=1.8.0",
    "fastapi>=0.111.0",
    "uvicorn[standard]>=0.30.0",
    "orjson",
    "tqdm",
    "requests",
    "openai>=1.0.0",
]

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", *packages])
print("Phase 1 dependencies installed.")

Phase 1 dependencies installed.


In [3]:
# cell 3
# Clone PyRAG. We will run it from the repo root through PYTHONPATH.
# The current repository snapshot may not include setup.py / pyproject.toml,
# so using PYTHONPATH is safer than pip install -e .

import os
import sys
import subprocess
from pathlib import Path

if not Path(REPO_DIR).exists():
    subprocess.check_call([
        "git", "clone",
        "https://github.com/GasolSun36/PyRAG.git",
        REPO_DIR
    ])
else:
    print("Repo already exists:", REPO_DIR)
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=False)

sys.path.insert(0, REPO_DIR)
os.environ["PYTHONPATH"] = REPO_DIR + ":" + os.environ.get("PYTHONPATH", "")

print("Repo ready:", REPO_DIR)

Repo ready: /content/PyRAG


In [4]:
# cell 4
# Define all input and output paths.
# Each dataset is fully separate: separate docs, separate questions, separate index.

from pathlib import Path

DATASETS = {
    "hotpotqa": {
        "questions": f"{PROJECT_DIR}/hotpotqa_dev_2017wiki_1000_converted.json",
        "docs": f"{PROJECT_DIR}/RAG/hotpotqa_docs_chunks.json",
        "index_dir": f"{INDEX_ROOT}/hotpotqa",
        "prepared_questions": f"{PREPARED_Q_DIR}/hotpotqa_shuffled.json",
        "final_evidence": f"{PYRAG_WORK_DIR}/hotpotqa_evidence.json",
    },
    "2wikimultihopqa": {
        "questions": f"{PROJECT_DIR}/2wikimultihopqa_dev_2020wiki_1000_converted.json",
        "docs": f"{PROJECT_DIR}/RAG/2wikimultihopqa_docs_chunks.json",
        "index_dir": f"{INDEX_ROOT}/2wikimultihopqa",
        "prepared_questions": f"{PREPARED_Q_DIR}/2wikimultihopqa_shuffled.json",
        "final_evidence": f"{PYRAG_WORK_DIR}/2wikimultihopqa_evidence.json",
    },
}

for name, cfg in DATASETS.items():
    print(f"\nDataset: {name}")
    for key in ["questions", "docs"]:
        path = Path(cfg[key])
        print(f"  {key}: {path} | exists={path.exists()} | size={path.stat().st_size if path.exists() else 'MISSING'}")
        if not path.exists():
            raise FileNotFoundError(f"Missing {key} file for {name}: {path}")

    Path(cfg["index_dir"]).mkdir(parents=True, exist_ok=True)

print("\nAll required input files exist.")


Dataset: hotpotqa
  questions: /content/drive/MyDrive/final_project/hotpotqa_dev_2017wiki_1000_converted.json | exists=True | size=157051600
  docs: /content/drive/MyDrive/final_project/RAG/hotpotqa_docs_chunks.json | exists=True | size=66820394

Dataset: 2wikimultihopqa
  questions: /content/drive/MyDrive/final_project/2wikimultihopqa_dev_2020wiki_1000_converted.json | exists=True | size=75086435
  docs: /content/drive/MyDrive/final_project/RAG/2wikimultihopqa_docs_chunks.json | exists=True | size=19865283

All required input files exist.


In [5]:
# cell 5
# Inspect schemas and confirm that we only need:
# - question from question files
# - Title and Text from docs chunk files

import json
from pathlib import Path

def load_json_array(path):
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    if not isinstance(data, list):
        raise ValueError(f"Expected a JSON array: {path}")
    return data

for name, cfg in DATASETS.items():
    questions = load_json_array(cfg["questions"])
    docs = load_json_array(cfg["docs"])

    print("=" * 80)
    print("Dataset:", name)
    print("Questions:", len(questions))
    print("Doc chunks:", len(docs))

    q0 = questions[0]
    d0 = docs[0]

    print("\nQuestion keys:", list(q0.keys()))
    print("Sample question:", q0.get("question"))
    print("Sample answer:", q0.get("answer"))
    print("Sample type:", q0.get("type"))

    print("\nDoc chunk keys:", list(d0.keys()))
    print("Sample title:", d0.get("Title"))
    print("Sample text preview:", str(d0.get("Text", ""))[:300])

    assert "question" in q0, f"{name}: question field is missing"
    assert "Title" in d0 and "Text" in d0, f"{name}: Title/Text fields are missing"

Dataset: hotpotqa
Questions: 1000
Doc chunks: 35029

Question keys: ['question', 'answer', 'type', 'titles', 'docs_chunks', 'docs', 'title_chunks', 'supports']
Sample question: What government position was held by the woman who portrayed Corliss Archer in the film Kiss and Tell?
Sample answer: Chief of Protocol
Sample type: bridge

Doc chunk keys: ['Chunk_id', 'Title', 'Paragraph_id', 'Text', 'Token_count']
Sample title: Meet Corliss Archer
Sample text preview: Meet Corliss Archer, a program from radio's Golden Age, ran from January 7, 1943 to September 30, 1956. Although it was CBS's answer to NBC's popular "A Date with Judy", it was also broadcast by NBC in 1948 as a summer replacement for "The Bob Hope Show". From October 3, 1952 to June 26, 1953, it ai
Dataset: 2wikimultihopqa
Questions: 1000
Doc chunks: 12685

Question keys: ['question', 'answer', 'type', 'titles', 'docs_chunks', 'docs', 'title_chunks', 'supports']
Sample question: Are North Marion High School (Oregon) and Seoul H

In [6]:
# cell 6
# Build a Search-R1/PyRAG-compatible corpus JSONL for each dataset.
# We only use Title and Text from each chunk.
# We also create shuffled question files with a fixed seed.

import json
import random
from pathlib import Path

SEED = 42

def clean_text(value):
    if value is None:
        return ""
    return str(value).strip()

def build_corpus_jsonl(dataset_name, cfg):
    docs = load_json_array(cfg["docs"])
    out_dir = Path(cfg["index_dir"])
    out_dir.mkdir(parents=True, exist_ok=True)

    corpus_jsonl = out_dir / "corpus.jsonl"
    seen = set()
    kept = 0
    skipped = 0

    with open(corpus_jsonl, "w", encoding="utf-8") as f:
        for i, item in enumerate(docs):
            title = clean_text(item.get("Title"))
            text = clean_text(item.get("Text"))

            if not title and not text:
                skipped += 1
                continue

            # Deduplicate exact repeated title-text pairs.
            dedup_key = (title, text)
            if dedup_key in seen:
                skipped += 1
                continue
            seen.add(dedup_key)

            chunk_id = clean_text(item.get("Chunk_id")) or f"{dataset_name}_chunk_{i:08d}"

            # PyRAG's HttpRetrievalAgent expects document["contents"] and splits it on newline:
            # first line = title, rest = body.
            contents = f"{title}\n{text}"

            record = {
                "id": chunk_id,
                "title": title,
                "text": text,
                "contents": contents,
            }
            f.write(json.dumps(record, ensure_ascii=False) + "\n")
            kept += 1

    cfg["corpus_jsonl"] = str(corpus_jsonl)
    print(f"{dataset_name}: wrote {kept:,} corpus records -> {corpus_jsonl}")
    print(f"{dataset_name}: skipped {skipped:,} empty/duplicate chunks")

def write_shuffled_questions(dataset_name, cfg):
    questions = load_json_array(cfg["questions"])

    # Keep the original index so Phase 2 can trace back to the original dataset row.
    for i, sample in enumerate(questions):
        sample["_pyrag_original_index"] = i

    rng = random.Random(SEED)
    rng.shuffle(questions)

    out_path = Path(cfg["prepared_questions"])
    out_path.parent.mkdir(parents=True, exist_ok=True)

    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(questions, f, ensure_ascii=False, indent=2)

    print(f"{dataset_name}: wrote shuffled questions -> {out_path}")

for name, cfg in DATASETS.items():
    build_corpus_jsonl(name, cfg)
    write_shuffled_questions(name, cfg)

hotpotqa: wrote 35,029 corpus records -> /content/drive/MyDrive/final_project/pyrag/indexes/hotpotqa/corpus.jsonl
hotpotqa: skipped 0 empty/duplicate chunks
hotpotqa: wrote shuffled questions -> /content/drive/MyDrive/final_project/pyrag/prepared_questions/hotpotqa_shuffled.json
2wikimultihopqa: wrote 12,685 corpus records -> /content/drive/MyDrive/final_project/pyrag/indexes/2wikimultihopqa/corpus.jsonl
2wikimultihopqa: skipped 0 empty/duplicate chunks
2wikimultihopqa: wrote shuffled questions -> /content/drive/MyDrive/final_project/pyrag/prepared_questions/2wikimultihopqa_shuffled.json


In [7]:
# cell 7
# Load the E5 embedding model.
# This follows the dense retriever style used by PyRAG/Search-R1.
# If you have the exact Search-R1 E5 checkpoint locally, replace EMBED_MODEL_NAME with that path.

import torch
from sentence_transformers import SentenceTransformer

EMBED_MODEL_NAME = "intfloat/e5-base-v2"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

embedder = SentenceTransformer(EMBED_MODEL_NAME, device=DEVICE)
embedder.max_seq_length = 512

print("Embedding model:", EMBED_MODEL_NAME)
print("Device:", DEVICE)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/67.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/650 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

Embedding model: intfloat/e5-base-v2
Device: cuda


In [8]:
# cell 8
# Build a FAISS cosine-similarity index for each dataset.
# Because E5 embeddings are normalized, IndexFlatIP is equivalent to cosine similarity.

import json
import time
import numpy as np
import faiss
from tqdm.auto import tqdm
from pathlib import Path

FORCE_REBUILD = False
EMBED_BATCH_SIZE = 128

def read_corpus_jsonl(path):
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records

def encode_passages(texts, batch_size=EMBED_BATCH_SIZE):
    # E5 expects passage/query prefixes.
    prefixed = [f"passage: {t}" for t in texts]
    embeddings = embedder.encode(
        prefixed,
        batch_size=batch_size,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=True,
    )
    return embeddings.astype("float32")

def build_index(dataset_name, cfg):
    index_dir = Path(cfg["index_dir"])
    index_path = index_dir / "index.faiss"
    meta_path = index_dir / "index_meta.json"

    if index_path.exists() and meta_path.exists() and not FORCE_REBUILD:
        print(f"{dataset_name}: index already exists, skipping. Set FORCE_REBUILD=True to rebuild.")
        return

    records = read_corpus_jsonl(cfg["corpus_jsonl"])
    if not records:
        raise ValueError(f"{dataset_name}: empty corpus")

    print(f"\nBuilding index for {dataset_name}: {len(records):,} chunks")
    t0 = time.time()

    dim = None
    index = None

    for start in tqdm(range(0, len(records), EMBED_BATCH_SIZE), desc=f"Indexing {dataset_name}"):
        batch = records[start:start + EMBED_BATCH_SIZE]
        texts = [r["contents"] for r in batch]
        emb = encode_passages(texts, batch_size=EMBED_BATCH_SIZE)

        if index is None:
            dim = emb.shape[1]
            index = faiss.IndexFlatIP(dim)

        index.add(emb)

    faiss.write_index(index, str(index_path))

    meta = {
        "dataset": dataset_name,
        "num_vectors": int(index.ntotal),
        "dimension": int(dim),
        "embedding_model": EMBED_MODEL_NAME,
        "corpus_jsonl": str(cfg["corpus_jsonl"]),
        "index_path": str(index_path),
        "created_unix_time": time.time(),
        "build_seconds": round(time.time() - t0, 2),
    }

    with open(meta_path, "w", encoding="utf-8") as f:
        json.dump(meta, f, ensure_ascii=False, indent=2)

    print(f"{dataset_name}: saved FAISS index -> {index_path}")
    print(f"{dataset_name}: saved metadata -> {meta_path}")

for name, cfg in DATASETS.items():
    build_index(name, cfg)


Building index for hotpotqa: 35,029 chunks


Indexing hotpotqa:   0%|          | 0/274 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2wikimultihopqa: saved FAISS index -> /content/drive/MyDrive/final_project/pyrag/indexes/2wikimultihopqa/index.faiss
2wikimultihopqa: saved metadata -> /content/drive/MyDrive/final_project/pyrag/indexes/2wikimultihopqa/index_meta.json


In [9]:
# cell 9
# Sanity-check retrieval from each separate index using a shuffled question.

import json
import faiss
import numpy as np
from pathlib import Path

def encode_queries(queries):
    prefixed = [f"query: {q}" for q in queries]
    embeddings = embedder.encode(
        prefixed,
        batch_size=32,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    )
    return embeddings.astype("float32")

def load_index_bundle(index_dir):
    index_dir = Path(index_dir)
    index = faiss.read_index(str(index_dir / "index.faiss"))
    corpus = read_corpus_jsonl(index_dir / "corpus.jsonl")
    return index, corpus

def search_index(index, corpus, query, topk=5):
    topk = min(topk, len(corpus))
    q_emb = encode_queries([query])
    scores, indices = index.search(q_emb, topk)

    hits = []
    for score, idx in zip(scores[0], indices[0]):
        if idx < 0:
            continue
        rec = corpus[int(idx)]
        hits.append({
            "score": float(score),
            "id": rec["id"],
            "title": rec["title"],
            "text": rec["text"],
        })
    return hits

for name, cfg in DATASETS.items():
    with open(cfg["prepared_questions"], "r", encoding="utf-8") as f:
        shuffled_questions = json.load(f)

    sample_question = shuffled_questions[0]["question"]
    index, corpus = load_index_bundle(cfg["index_dir"])
    hits = search_index(index, corpus, sample_question, topk=5)

    print("=" * 80)
    print("Dataset:", name)
    print("Sample question:", sample_question)
    print("\nTop-5 retrieved chunks:")
    for i, h in enumerate(hits, 1):
        print(f"\n[{i}] score={h['score']:.4f} | title={h['title']}")
        print(h["text"][:500])

Dataset: hotpotqa
Sample question: Which musical fantasy film is older, Bedknobs and Broomsticks or The Muppet Christmas Carol?

Top-5 retrieved chunks:

[1] score=0.8881 | title=Bedknobs and Broomsticks
Bedknobs and Broomsticks is a 1971 British-American musical fantasy film produced by Walt Disney Productions and released by Buena Vista Distribution Company in North America on December 13, 1971. It is based upon the books "The Magic Bedknob; or, How to Become a Witch in Ten Easy Lessons" (1943) and "Bonfires and Broomsticks" (1945) by English children's author Mary Norton. The film, which combines live action and animation, stars Angela Lansbury and David Tomlinson.
The film is frequently comp

[2] score=0.8509 | title=The Muppet Christmas Carol
The Muppet Christmas Carol is a 1992 American-British musical fantasy comedy-drama film and an adaptation of Charles Dickens's 1843 novel "A Christmas Carol". It is the fourth in a series of live-action musical films featuring The Muppets, wi

In [10]:
# cell 10
# Write a PyRAG-compatible local E5 retriever server.
# It implements POST /retrieve with:
# {
#   "queries": [...],
#   "topk": 5,
#   "return_scores": true
# }
#
# Response:
# {
#   "result": [[{"document": {"id": ..., "contents": ...}, "score": ...}, ...]]
# }

from pathlib import Path
import textwrap

RETRIEVER_SERVER_SCRIPT = f"{CUSTOM_SCRIPTS_DIR}/local_e5_retriever_server.py"

server_code = r'''
import argparse
import json
from pathlib import Path
from typing import List

import faiss
import numpy as np
import uvicorn
from fastapi import FastAPI
from pydantic import BaseModel
from sentence_transformers import SentenceTransformer


class RetrieveRequest(BaseModel):
    queries: List[str]
    topk: int = 5
    return_scores: bool = True


def read_corpus_jsonl(path: Path):
    corpus = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                corpus.append(json.loads(line))
    return corpus


def create_app(index_dir: str, model_name: str, device: str):
    index_dir = Path(index_dir)
    index_path = index_dir / "index.faiss"
    corpus_path = index_dir / "corpus.jsonl"

    if not index_path.exists():
        raise FileNotFoundError(f"Missing FAISS index: {index_path}")
    if not corpus_path.exists():
        raise FileNotFoundError(f"Missing corpus file: {corpus_path}")

    index = faiss.read_index(str(index_path))
    corpus = read_corpus_jsonl(corpus_path)

    if index.ntotal != len(corpus):
        raise ValueError(f"Index size {index.ntotal} != corpus size {len(corpus)}")

    embedder = SentenceTransformer(model_name, device=device)
    embedder.max_seq_length = 512

    app = FastAPI(title="Local E5 Retriever for PyRAG")

    def encode_queries(queries: List[str]) -> np.ndarray:
        prefixed = [f"query: {q}" for q in queries]
        embeddings = embedder.encode(
            prefixed,
            batch_size=32,
            convert_to_numpy=True,
            normalize_embeddings=True,
            show_progress_bar=False,
        )
        return embeddings.astype("float32")

    @app.get("/health")
    def health():
        return {
            "status": "ok",
            "num_documents": len(corpus),
            "index_vectors": int(index.ntotal),
            "model_name": model_name,
            "device": device,
        }

    @app.post("/retrieve")
    def retrieve(req: RetrieveRequest):
        topk = max(1, min(int(req.topk), len(corpus)))
        query_emb = encode_queries(req.queries)
        scores, indices = index.search(query_emb, topk)

        all_results = []
        for row_scores, row_indices in zip(scores, indices):
            hits = []
            for score, idx in zip(row_scores, row_indices):
                if int(idx) < 0:
                    continue

                rec = corpus[int(idx)]
                document = {
                    "id": rec.get("id", str(idx)),
                    "contents": rec["contents"],
                    "title": rec.get("title", ""),
                    "text": rec.get("text", ""),
                }

                if req.return_scores:
                    hits.append({
                        "document": document,
                        "score": float(score),
                    })
                else:
                    hits.append(document)

            all_results.append(hits)

        return {"result": all_results}

    return app


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--index_dir", required=True)
    parser.add_argument("--model_name", default="intfloat/e5-base-v2")
    parser.add_argument("--device", default="cuda")
    parser.add_argument("--host", default="127.0.0.1")
    parser.add_argument("--port", type=int, default=8008)
    args = parser.parse_args()

    app = create_app(
        index_dir=args.index_dir,
        model_name=args.model_name,
        device=args.device,
    )

    uvicorn.run(app, host=args.host, port=args.port)


if __name__ == "__main__":
    main()
'''

Path(RETRIEVER_SERVER_SCRIPT).write_text(server_code, encoding="utf-8")
print("Wrote retriever server:", RETRIEVER_SERVER_SCRIPT)

Wrote retriever server: /content/drive/MyDrive/final_project/pyrag/custom_scripts/local_e5_retriever_server.py


In [11]:
# cell 11
# Optional sanity launch for one dataset.
# Keep ACTIVE_DATASET as "hotpotqa" or "2wikimultihopqa".
# In Phase 2, we will start the server for each dataset separately before running PyRAG.

import time
import requests
import subprocess
import sys
from pathlib import Path

ACTIVE_DATASET = "hotpotqa"  # change to "2wikimultihopqa" for the other index
RETRIEVER_PORT = 8008

# Stop a previous server process from this notebook, if it exists.
if "retriever_proc" in globals() and retriever_proc.poll() is None:
    retriever_proc.terminate()
    time.sleep(3)

log_path = f"{PYRAG_WORK_DIR}/retriever_{ACTIVE_DATASET}.log"
log_file = open(log_path, "w", encoding="utf-8")

cmd = [
    sys.executable,
    RETRIEVER_SERVER_SCRIPT,
    "--index_dir", DATASETS[ACTIVE_DATASET]["index_dir"],
    "--model_name", EMBED_MODEL_NAME,
    "--device", DEVICE,
    "--host", "127.0.0.1",
    "--port", str(RETRIEVER_PORT),
]

retriever_proc = subprocess.Popen(
    cmd,
    stdout=log_file,
    stderr=subprocess.STDOUT,
)

print("Started retriever process PID:", retriever_proc.pid)
print("Log file:", log_path)

# Wait until the server is ready.
for attempt in range(60):
    try:
        r = requests.get(f"http://127.0.0.1:{RETRIEVER_PORT}/health", timeout=2)
        if r.status_code == 200:
            print("Retriever health:", r.json())
            break
    except Exception:
        time.sleep(2)
else:
    raise RuntimeError("Retriever did not become ready. Check the log file.")

Started retriever process PID: 3272
Log file: /content/drive/MyDrive/final_project/pyrag/retriever_hotpotqa.log
Retriever health: {'status': 'ok', 'num_documents': 35029, 'index_vectors': 35029, 'model_name': 'intfloat/e5-base-v2', 'device': 'cuda'}


In [12]:
# cell 12
# Test the HTTP /retrieve endpoint exactly like PyRAG will call it.

import json
import requests

with open(DATASETS[ACTIVE_DATASET]["prepared_questions"], "r", encoding="utf-8") as f:
    shuffled_questions = json.load(f)

test_question = shuffled_questions[0]["question"]

payload = {
    "queries": [test_question],
    "topk": 5,
    "return_scores": True,
}

response = requests.post(
    f"http://127.0.0.1:{RETRIEVER_PORT}/retrieve",
    json=payload,
    timeout=60,
)
response.raise_for_status()
result = response.json()

print("Question:", test_question)
print("Number of hits:", len(result["result"][0]))

for i, hit in enumerate(result["result"][0], 1):
    doc = hit["document"]
    print("=" * 80)
    print(f"Hit {i} | score={hit['score']:.4f}")
    print("Title:", doc.get("title"))
    print("Contents preview:", doc["contents"][:700])

Question: Which musical fantasy film is older, Bedknobs and Broomsticks or The Muppet Christmas Carol?
Number of hits: 5
Hit 1 | score=0.8881
Title: Bedknobs and Broomsticks
Contents preview: Bedknobs and Broomsticks
Bedknobs and Broomsticks is a 1971 British-American musical fantasy film produced by Walt Disney Productions and released by Buena Vista Distribution Company in North America on December 13, 1971. It is based upon the books "The Magic Bedknob; or, How to Become a Witch in Ten Easy Lessons" (1943) and "Bonfires and Broomsticks" (1945) by English children's author Mary Norton. The film, which combines live action and animation, stars Angela Lansbury and David Tomlinson.
The film is frequently compared with "Mary Poppins" (1964), since it combines live action and animation and is partially set in the streets of London. It also features numerous cast members from "Mary P
Hit 2 | score=0.8509
Title: The Muppet Christmas Carol
Contents preview: The Muppet Christmas Carol
The Mup

In [13]:
# cell 13
# Save a Phase 1 manifest.
# Phase 2 will read this manifest to know which index and shuffled question file to use.

import json
import time
from pathlib import Path

manifest = {
    "phase": "phase_1_prepare_retriever_indexes",
    "created_unix_time": time.time(),
    "recommended_full_pipeline_gpu": "A100 80GB",
    "embedding_model": EMBED_MODEL_NAME,
    "retriever_server_script": RETRIEVER_SERVER_SCRIPT,
    "default_topk": 5,
    "adaptive_retry_topk": 10,
    "datasets": DATASETS,
    "notes": [
        "Retrieval uses only Title and Text from docs chunk files.",
        "PyRAG input uses only sample['question']; gold supports/titles/docs are not used for retrieval.",
        "Each dataset has a separate corpus and FAISS index.",
        "Questions were shuffled with a fixed seed.",
    ],
}

manifest_path = f"{PYRAG_WORK_DIR}/phase1_manifest.json"
with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2)

print("Saved Phase 1 manifest:", manifest_path)
print("\nPhase 1 is complete.")
print("Next phase: start the correct retriever index, run the two vLLM servers, then run PyRAG per shuffled question.")

Saved Phase 1 manifest: /content/drive/MyDrive/final_project/pyrag/phase1_manifest.json

Phase 1 is complete.
Next phase: start the correct retriever index, run the two vLLM servers, then run PyRAG per shuffled question.
